In [1]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
import os
%matplotlib qt

frame_num = 370
cam = 0
image_path = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/'
dict_path  = 'D:/Documents/data_for_gs/mov1_2023_08_09_60ms/dict/frames_model.pkl'
path_output = 'D:/Documents/gaussian_model_output/'
interest_point_h5_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evalutation'


image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov128_2023_08_09_60ms/'
dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/dict/frames_model_evaluation.pkl'
# path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

# image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'
# dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model_evaluation.pkl'
# path_output = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

model_name = 'fly_features_compare'
file_name = 'fly_model'


model_name = 'fly_features_dense'
file_name = 'fly_model'

idx_iter = 0
model_name = 'fly_angle_sweeps_roll_yaw_nom_all_frames_1000'
file_name = f'fly_model_scale_iter{idx_iter}'
# model_name = 'fly_features_3cam'
# file_name = 'fly_model_aa'
iteration = 1
frame0 = 1650#1620
input_dir = f'{path_output}/{model_name}'
num_iter = 2


with open(f'{input_dir}/results/{frame0}/{file_name}_results.pkl', 'rb') as handle:
    output_angles_weights = pickle.load(handle)
    

with open(dict_path,'rb') as f:
    frames = pickle.load(f)

# wakk = [output_angles_weights['weights'][idx] for idx in range(idx_iter,len(output_angles_weights['weights']),num_iter)]
# weights= {}
# weights['weights'] = wakk



input_dir_ini = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation'

with open(f'{input_dir_ini}/nominal_initial_angles.pkl', 'rb') as handle:
    nominal_initial_angles = pickle.load(handle)



weights = output_angles_weights['weights'][0]


In [477]:
mov_frame = 'mov_128_frame_839'
mov = int(mov_frame.split('_')[1]) 
frame = int(mov_frame.split('_')[3]) 


path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}/wing1_gt_points_frame{frame}.pkl'
with open(path,'rb') as f:
    ini_angles1 = pickle.load(f)
cam1 = np.vstack(ini_angles1['cam1'])
plt.scatter(cam1[:,0],cam1[:,1])
path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}/wing2_gt_points_frame{frame}.pkl'
with open(path,'rb') as f:
    ini_angles2 = pickle.load(f)
cam1 = np.vstack(ini_angles2['cam1'])
plt.scatter(cam1[:,0],cam1[:,1])

In [506]:
def open_file(path,ang_dict = {}):
    with open(path,'rb') as f:
        ini_angles = pickle.load(f)

    ang_dict = {ang_name: angles for ang_name,angles in ini_angles.items()}

    return ang_dict

frame = 1620
path = f'D:/Documents/gaussian_model_output/fly_angle_sweeps_roll_yaw_nom/{frame}/initial/'
dir_names = [name for name in os.listdir(path) if os.path.isfile(os.path.join(path, name))]
ini_angles = {idx:open_file(f'{path}/{dir}') for idx,dir in enumerate(dir_names)}

bod = np.vstack([ini_angles[idx]['body_angles'] for idx in range(len(ini_angles))])





In [5]:
mov_frame

['mov_36_frame_865']

In [ ]:
from Evaluation import Evaluation
mov_frame = list(nominal_initial_angles.keys())[6]
mov = int(mov_frame.split('_')[1]) 
frame0 = int(mov_frame.split('_')[3]) 

    
interest_points_path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}'
image_path =  f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{mov}_2023_08_09_60ms/'
frame = Evaluation(interest_points_path,image_path,frame0,input_dir,output_angles_weights,frame0,iteration,file_name,frames_dict = frames)

In [ ]:
min_angles = []
min_frames = []
min_dist = []


def calc_dist_frame(interest_points_path,image_path,frame_num,input_dir,output_angles_weights,frame0,iteration,file_name, frames):
    
    frame = FlyOutput(image_path,frame_num,input_dir,output_angles_weights,frame0,iteration,file_name,frames_dict = frames)
    frame.intersect_projections

    
    interest_points = frame.load_parimiter(interest_points_path)
    frame.add_interest_points_and_triangulate(interest_points)
    frame.define_wings_interest_points()
    frame.calc_wing_le_te(num_of_bins = 20,perc_wing_for_le = 1,wing_length_snip = 0.15)
    fitted_gs,fitted_interest = frame.fit_interest_and_gs()
    # frame.calculate_dist_interest_gs()
    return frame,fitted_gs,fitted_interest
    frames_list.append(frame)


all_frames = []
all_dist = []
all_dist3d = []

for mov_frame in list(nominal_initial_angles.keys())[6:7]:
    mov = int(mov_frame.split('_')[1]) 
    frame0 = int(mov_frame.split('_')[3]) 
    dist_mean = []
    dist2d_mean = []
    for idx_iter in range(8):
        
        
        file_name = f'fly_model_scale_iter{idx_iter}'
        # model_name = 'fly_features_3cam'
        # file_name = 'fly_model_aa'
        iteration = 1000
        input_dir = f'{path_output}/{model_name}'


        with open(f'{input_dir}/results/{frame0}/{file_name}_results.pkl', 'rb') as handle:
            output_angles_weights = pickle.load(handle)
            



        def intersect_all_cams(frames,cam,intersected,tol = 1):
            for cam in range(4):
                intersected = Utils.intersection_per_cam(frames, cam, intersected, tol=tol) 
            return intersected

        frames_list=[]
       
        for frame_num in range(frame0,frame0+1,3):

            interest_points_path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}'
            image_path =  f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{mov}_2023_08_09_60ms/'
            frame,fitted_gs,fitted_interest = calc_dist_frame(interest_points_path,image_path,frame_num,input_dir,output_angles_weights,frame0,iteration,file_name, frames)
            frames_list.append(frame)
            all_frames.append(frame)

        # prev_length = 0 
        # frames_interest_points = {}
        # for key in interest_points.keys():
        #     frames_interest_points[key] = list(range(prev_length,prev_length + len(interest_points[key]))) 
        #     prev_length += len(interest_points[key])

            # dist_mean.append(frame.dist_3d_gs_to_interest.flatten()*1000)
            # dist2d_mean.append(frame.dist_2d_interest_to_gs.flatten())
    all_dist.append(dist2d_mean)
    all_dist3d.append(dist_mean)

    # min_dist.append(np.min(dist2d_mean))
    # min_frames.append(np.argmin(dist2d_mean))
    # min_angles.append(np.argmin(dist2d_mean))


In [67]:
rwing_poins = np.vstack([frame.right_wing_le,frame.right_wing_te])

rwing_poins.shape

(68, 3)

In [68]:
from scipy.signal import savgol_filter


frame = all_frames[1]

rwing_poins = np.vstack([frame.right_wing_le,frame.right_wing_te])
lwing_poins = np.vstack([frame.left_wing_le,frame.left_wing_te])
lwing_poins = np.unique(lwing_poins,axis = 0)
rwing_poins = np.unique(rwing_poins,axis = 0)


interest_right_wing = frame.rotated_points_3d[frame.interest_right_wing_boundry,:]
interest_left_wing = frame.rotated_points_3d[frame.interest_left_wing_boundry,:]


indices_right_wing_bound = frame.cyclic_sort(rwing_poins,frame.right_wing_span,frame.right_wing_chord)
indices_left_wing_bound = frame.cyclic_sort(lwing_poins,frame.left_wing_span,frame.left_wing_chord)
indices_interest_right_wing = frame.cyclic_sort(interest_right_wing,frame.right_wing_span,frame.right_wing_chord)
indices_interest_left_wing = frame.cyclic_sort(interest_left_wing,frame.left_wing_span,frame.left_wing_chord)
left_bound = lwing_poins[indices_left_wing_bound]
right_bound = rwing_poins[indices_right_wing_bound]


# left_bound = np.vstack([savgol_filter(points, 7, 3, mode='nearest') for points in left_bound.T]).T
# right_bound = np.vstack([savgol_filter(points, 7, 3, mode='nearest') for points in right_bound.T]).T


fig = go.Figure()
Plotters.scatter3d(fig,interest_right_wing,'red',3,'intr',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,right_bound,'magenta',2,'right wing',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,left_bound,'blue',2,'left wing',show_colorbar = False,mode='lines + markers')
Plotters.scatter3d(fig,interest_left_wing,'cyan',3,'intl',show_colorbar = False)

Plotters.scatter3d(fig,frame.body,'purple',2,'right wing',show_colorbar = False)


In [88]:
np.sum([np.sqrt(np.sum(points_for_line[idx]**2 - points_for_dist**2)) for idx in [0,1]])

C:\Users\Roni\AppData\Local\Temp\ipykernel_14676\589375274.py:1: RuntimeWarning:

invalid value encountered in sqrt



nan

AxisError: axis 1 is out of bounds for array of dimension 1

In [101]:
k = 1
all_dist = []
for idx in range(interest_left_wing.shape[0]):
    points_for_dist = interest_left_wing[idx]
    dist_list = []
    for k in range(left_bound.shape[0] - 1):
        points_for_line = left_bound[k:k+2]
        dist = np.sum([np.sqrt(np.sum((points_for_line[idx] - points_for_dist)**2)) for idx in [0,1]])
       

        dist_list.append(np.linalg.norm(dist))
    all_dist.append(np.argmin(dist_list))
all_dist



fig = go.Figure()
Plotters.scatter3d(fig,interest_right_wing,'red',3,'intr',show_colorbar = False)
Plotters.scatter3d(fig,right_bound,'magenta',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,left_bound,'blue',2,'left wing',show_colorbar = False, mode='lines + markers')
Plotters.scatter3d(fig,interest_left_wing,'cyan',3,'intl',show_colorbar = False)

Plotters.scatter3d(fig,frame.body,'purple',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[all_dist],'green',3,'start',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[np.array(all_dist) + 1],'red',3,'end',show_colorbar = False)

fig.show()

In [138]:

line =       point_line - origin # the line - a vector
point_to_origin = point - origin # a vector from the point to the lines origin
point_line_projected_on_line = np.dot(line, point_to_origin) # project the vector from the origin to the point on the line
line_sq_length = np.dot(point_to_origin, point_to_origin) # project the vector from the origin to the point on the line

t = np.dot(point_line_projected_on_line, line) / line_sq_length
point_line_projected_on_line

-1.3201116481201034e-08

In [145]:
point = interest_left_wing[2]
dist = np.argmin([point_to_segment_projection(point, left_bound[k], left_bound[k+1]) for k in range(left_bound.shape[0] - 1)])
dist

32

In [157]:
all_dist

[5,
 5,
 5,
 5,
 6,
 6,
 6,
 6,
 6,
 7,
 7,
 7,
 7,
 8,
 8,
 8,
 8,
 8,
 9,
 9,
 11,
 11,
 11,
 11,
 11,
 10,
 10,
 10,
 12,
 12,
 12,
 12,
 12,
 1,
 1,
 0,
 1,
 0,
 0,
 2,
 2,
 2,
 2,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 5,
 5,
 5,
 5]

In [160]:
left_bound[dist_closest_gs_to_interest]

array([[-0.00203345, -0.00731631,  0.00130322],
       [-0.00203915, -0.00685589,  0.00125938],
       [-0.00200441, -0.00767739,  0.00137054],
       [-0.00214899, -0.00646649,  0.00124359],
       [-0.00229991, -0.00621184,  0.00119061],
       [-0.00254854, -0.0060471 ,  0.00099333],
       [-0.00276112, -0.00615947,  0.000861  ],
       [-0.00293349, -0.00642134,  0.00078292],
       [-0.00289493, -0.00691814,  0.0007008 ],
       [-0.00275709, -0.00753988,  0.00070033],
       [-0.00248582, -0.00793388,  0.00097785],
       [-0.00231264, -0.00807728,  0.00109082],
       [-0.0026276 , -0.00771641,  0.00078065],
       [-0.00231264, -0.00807728,  0.00109082]])

In [208]:
def point_to_segment_projection(point, origin, point_line):
    line = point_line - origin # the line - a vector
    point_to_origin = point - origin # a vector from the point to the lines origin
    line_sq_length = np.dot(line, line) # project the vector from the origin to the point on the line
    t = np.dot(point_to_origin, line) / line_sq_length
    if 0 <= t <= 1:
        projection = origin + t * line
        dist = np.linalg.norm(point - projection)
        return dist
    else:
        return float('inf')
dist_closest_interest_to_gs = []

for idx in range(interest_left_wing.shape[0]):
    dist_closest_interest_to_gs.append(np.argmin([point_to_segment_projection(interest_left_wing[idx], left_bound[k], left_bound[k+1]) for k in range(left_bound.shape[0] - 1)]))

dist_closest_gs_to_interest = []
for idx in range(left_bound.shape[0]):
    dist_closest_gs_to_interest.append(np.argmin([point_to_segment_projection(left_bound[idx], interest_left_wing[k], interest_left_wing[k+1]) for k in range(interest_left_wing.shape[0] - 1)]))

# now we need to find the 3d point on the closest line - to calculate the 2d distance


def project_point_on_line(points_to_project_on_line,line_points,dist_closest_interest_to_gs):

    points_of_line = line_points[dist_closest_interest_to_gs:dist_closest_interest_to_gs + 2]
    line = (points_of_line[1] - points_of_line[0])/np.linalg.norm((points_of_line[1] - points_of_line[0]))
    return np.dot(points_to_project_on_line - points_of_line[0],line)*line + points_of_line[0]

interest_on_bound_line = [project_point_on_line(interest_left_wing[idx],left_bound,dist_closest_interest_to_gs[idx]) for idx in range(len(dist_closest_interest_to_gs))]
bound_line_on_interest = [project_point_on_line(left_bound[idx],interest_left_wing,dist_closest_gs_to_interest[idx]) for idx in range(len(dist_closest_gs_to_interest))]


fig = go.Figure()
Plotters.scatter3d(fig,interest_right_wing,'red',3,'intr',show_colorbar = False)
Plotters.scatter3d(fig,right_bound,'magenta',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,left_bound,'blue',2,'left wing',show_colorbar = False, mode='lines + markers')


Plotters.scatter3d(fig,frame.body,'purple',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,interest_left_wing,'black',3,'start',show_colorbar = False)
# Plotters.scatter3d(fig,interest_left_wing[np.array(dist_closest_interest_to_gs) + 1],'red',3,'end',show_colorbar = False)

Plotters.scatter3d(fig,np.vstack(bound_line_on_interest),'crimson',5,'end',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[dist_closest_interest_to_gs],'green',3,'start',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[np.array(dist_closest_interest_to_gs) + 1],'red',3,'end',show_colorbar = False)


fig.show()

In [198]:
interest_left_wing

array([[-0.00203576, -0.0072656 ,  0.00128979],
       [-0.00204613, -0.00680699,  0.0012384 ],
       [-0.00202673, -0.00759722,  0.00130444],
       [-0.00217612, -0.00643269,  0.00120393],
       [-0.00236728, -0.00612965,  0.00116902],
       [-0.00259326, -0.00600119,  0.00105537],
       [-0.00278031, -0.00619714,  0.00090658],
       [-0.00291647, -0.00657867,  0.00082011],
       [-0.00290274, -0.00704798,  0.00083858],
       [-0.00277621, -0.00756528,  0.00087219],
       [-0.00245141, -0.00787191,  0.00099923],
       [-0.00227863, -0.00799726,  0.00118861],
       [-0.00263721, -0.00772213,  0.0008363 ],
       [-0.00197819, -0.00801942,  0.00131394]])

In [205]:
all_points = []


idx_of_line_points = dist_closest_interest_to_gs
line_points = left_bound
points_to_project_on_line = interest_left_wing


def project_point_on_line(points_to_project_on_line,line_points,dist_closest_interest_to_gs):

    points_of_line = line_points[dist_closest_interest_to_gs:dist_closest_interest_to_gs + 2]
    line = (points_of_line[1] - points_of_line[0])/np.linalg.norm((points_of_line[1] - points_of_line[0]))
    return np.dot(points_to_project_on_line - points_of_line[0],line)*line + points_of_line[0]

all_points = [project_point_on_line(points_to_project_on_line[idx],line_points,dist_closest_interest_to_gs[idx]) for idx in range(len(idx_of_line_points))]
    # closest_points_gs_to_interest = points_to_project_on_line[idx]
    # points_of_line = line_points[dist_closest_interest_to_gs[idx]:dist_closest_interest_to_gs[idx] + 2]
    # line = (points_of_line[1] - points_of_line[0])/np.linalg.norm((points_of_line[1] - points_of_line[0]))
    # point_on_the_line = np.dot(closest_points_gs_to_interest - points_of_line[0],line)*line + points_of_line[0]
    # all_points.append(point_on_the_line)

In [123]:
k = 1
all_dist = []
for idx in range(interest_left_wing.shape[0]):
    points_for_dist = interest_left_wing[idx]
    dist_list = []
    for k in range(left_bound.shape[0] - 1):
        points_for_line = left_bound[k:k+2]

        line = points_for_line[0] - points_for_line[1]
        line = line / np.linalg.norm(line)
        point_to_origin = points_for_line[0] - points_for_dist
        projected = np.dot(point_to_origin,line)
        if projected > 0 and projected < 1:
            dist = point_to_origin - (line*np.atleast_2d(projected).T)
        else:
            dist = 9999
        dist_list.append(np.linalg.norm(dist))
    all_dist.append(np.argmin(dist_list))
all_dist



fig = go.Figure()
Plotters.scatter3d(fig,interest_right_wing,'red',3,'intr',show_colorbar = False)
Plotters.scatter3d(fig,right_bound,'magenta',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,left_bound,'blue',2,'left wing',show_colorbar = False, mode='lines + markers')
Plotters.scatter3d(fig,interest_left_wing[3:4],'cyan',3,'intl',show_colorbar = False)

Plotters.scatter3d(fig,frame.body,'purple',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[all_dist][3:4],'green',3,'start',show_colorbar = False)
Plotters.scatter3d(fig,left_bound[np.array(all_dist) + 1][3:4],'red',3,'end',show_colorbar = False)

fig.show()

In [53]:
all_dist

[44, 45, 48, 41, 33, 18, 11, 0, 1, 17, 29, 34, 22, 52]

In [ ]:


rwing_poins = frame.right_wing_le    
rwing_poins_te = frame.right_wing_te    

indices_right_wing_bound = frame.cyclic_sort(rwing_poins,frame.right_wing_span,frame.right_wing_chord)
indices_left_wing_bound = frame.cyclic_sort(rwing_poins_te,frame.right_wing_span,frame.right_wing_chord)


plt.plot(rwing_poins[indices_right_wing_bound,0],rwing_poins[indices_right_wing_bound,1],'*')
plt.plot(rwing_poins_te[indices_left_wing_bound,0],rwing_poins_te[indices_left_wing_bound,1],'*')


IndexError: index 56 is out of bounds for axis 0 with size 39

In [3]:
fig = go.Figure()
Plotters.scatter3d(fig,fitted_gs,'green',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,fitted_interest,'purple',2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing,'blue',2,'left wing',show_colorbar = False)
# Plotters.scatter3d(fig,frame.rotated_points_3d[7:8,:],'blue',5,'interest',show_colorbar = False)
# # Plotters.scatter3d(fig,frames_list[frame-frame0].gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_te,'cyan',4,'te',show_colorbar = False)

Plotters.scatter3d(fig,frame.left_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing_te,'cyan',4,'te',show_colorbar = False)
fig.show()


In [6]:
from scipy.signal import savgol_filter
wing_bound = np.vstack((frame.left_wing_le,frame.left_wing_te))
wing_bound2 = np.vstack((frame.right_wing_le,frame.right_wing_te))

left_bound,right_bound,interest_left_wing,interest_right_wing = frame.zsocre_ol_calc_indices()

fig = go.Figure()
Plotters.scatter3d(fig,left_bound,'green',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,wing_bound,'blue',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,right_bound,'magenta',3,'body',show_colorbar = False)

Plotters.scatter3d(fig,wing_bound2,'red',3,'body',show_colorbar = False)

Plotters.scatter3d(fig,interest_right_wing,'black',5,'black',show_colorbar = False)
Plotters.scatter3d(fig,interest_left_wing,'black',5,'black',show_colorbar = False)

fig.show()


In [94]:
def closest_dist_to_line(bound,interest):
    line = bound[0] - bound[1]
    line = line / np.linalg.norm(line)
    point_to_origin = interest - bound[0]
    projected = np.dot(point_to_origin,line)
    dist = point_to_origin - (line*np.atleast_2d(projected).T)
    
    return np.linalg.norm(dist)

idx = 1

[closest_dist_to_line(left_bound[k:k+2],interest_right_wing[idx]) for k in range(left_bound.shape[0] - 1)]
idx = 8
for idx in range(interest_right_wing.shape[0]):
    print(np.argmin([closest_dist_to_line(left_bound[k:k+2],interest_right_wing[idx]) for k in range(left_bound.shape[0] - 1)]))


3
3
3
3
3
3
3
3
3
3
3
3
3
3


C:\Users\Roni\AppData\Local\Temp\ipykernel_88608\1481370837.py:3: RuntimeWarning:

invalid value encountered in divide



In [97]:
fig = go.Figure()
Plotters.scatter3d(fig,left_bound[0:30,:],'green',5,'body',show_colorbar = False)
Plotters.scatter3d(fig,interest_right_wing,'blue',3,'body',show_colorbar = False)



fig.show()

In [79]:
interest_left_wing[2]

array([-0.00192452, -0.01101843,  0.0010584 ])

In [61]:
def closest_dist_to_line(left_bound,interest):
    line = left_bound[0] - left_bound[1]
    line = line / np.linalg.norm(line)
    point_to_origin = interest - left_bound[0]
    projected = np.dot(point_to_origin,line)
    dist = point_to_origin - (line*np.atleast_2d(projected).T)
    
    return np.linalg.norm(dist)

min_bound = []
for idx in range(interest_left_wing.shape[0]-1):
    mindist = [closest_dist_to_line(left_bound[k:k+2],interest_left_wing[idx]) for k in range(left_bound.shape[0]-1)]
    k = np.argmin(mindist)
    min_bound.append(left_bound[k:k+2])

pltmin = np.vstack(min_bound)

C:\Users\Roni\AppData\Local\Temp\ipykernel_88608\1752287838.py:3: RuntimeWarning:

invalid value encountered in divide



In [62]:
pltmin

array([[-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.00079767],
       [-0.00284897, -0.00632019,  0.000

In [52]:
mindist

[0.003382569126475693,
 0.0024839284268598407,
 0.003050179610473372,
 nan,
 0.0023791328632779764,
 0.0033986984787962742,
 0.00309079387159069,
 0.0020589326864114514,
 0.003203390821180357,
 nan,
 0.0019514596797054846,
 0.0006531698837975294,
 0.0007750082138120979,
 0.0013034962814458895,
 0.002733596947291951,
 0.0012244525614249168,
 0.0016135141876601215,
 0.0006849361595582024,
 nan,
 0.00049137897625056,
 0.00037475822830957474,
 0.002092977366556479,
 0.0004820321491025856,
 0.0005181592110776147,
 0.0008354677319443297,
 0.0006316530780137955,
 0.0015460610426534576,
 0.0012063185821165122,
 0.0003776108443948705,
 nan,
 0.002235796912408634,
 0.00039477470156365935,
 0.0008180157470237544,
 0.0005451615875807626,
 0.0010136614177003326,
 0.0016309838928527905,
 0.001016422474379437,
 0.0007058677225969545,
 0.0006843815525574625,
 0.0008560516477883727,
 0.0025153287769325377,
 0.003426323598580518,
 0.002893616265798418,
 0.0012958220845254765,
 0.0028251451778811406,
 0.

In [42]:

def closest_dist_to_line(interest_left_wing,wing_bound):
    line = interest_left_wing[0] - interest_left_wing[1]
    line = line / np.linalg.norm(line)
    point_to_origin = wing_bound - interest_left_wing[0]
    projected = np.dot(point_to_origin,line)
    dist = point_to_origin - (line*np.atleast_2d(projected).T)
    min_idx = np.argmin(np.linalg.norm(dist,axis = 1))
    return wing_bound[min_idx],interest_left_wing[0] + line*np.atleast_2d(projected[min_idx]).T





clos = []
inte = []
for k in range(interest_left_wing.shape[0]-1):
    closest,interest = closest_dist_to_line(interest_right_wing[k:k+2],left_bound) 
    clos.append(closest)
    inte.append(interest)





In [15]:
np.vstack(inte)

array([[-0.32858041, -0.32831639, -0.32863388],
       [-0.32858696, -0.32833475, -0.32863804],
       [-0.32861234, -0.32840587, -0.32865416],
       ...,
       [ 0.23678314,  0.23853634,  0.23703617],
       [ 0.23678858,  0.23851413,  0.23703762],
       [ 0.23678403,  0.23853271,  0.23703641]])

In [43]:
fig = go.Figure()
Plotters.scatter3d(fig,np.vstack(clos),'green',5,'body',show_colorbar = False)
Plotters.scatter3d(fig,interest_right_wing,'blue',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,np.vstack(inte),'black',5,'body',show_colorbar = False)
Plotters.scatter3d(fig,pltmin,'magenta',5,'body',show_colorbar = False)

Plotters.scatter3d(fig,left_bound,'red',3,'body',show_colorbar = False)

fig.show()

In [4]:
[plt.hist(ang, alpha = 0.5, bins = 50) for ang in dist2d_mean[0:1]]

[(array([535., 605., 591., 379., 253., 185., 149., 103.,  62.,  37.,  30.,
          35.,  19.,  15.,  13.,  15.,  10.,  15.,  13.,  25.,  31.,  25.,
          20.,  19.,  16.,  17.,  16.,  16.,  15.,  15.,  66.,  55.,  42.,
          21.,  16.,  14.,  13.,  11.,  14.,  12.,   8.,   8.,   6.,   5.,
           5.,   5.,   5.,   6.,   4.,   5.]),
  array([2.01329585e-02, 4.65431175e-01, 9.10729392e-01, 1.35602761e+00,
         1.80132582e+00, 2.24662404e+00, 2.69192226e+00, 3.13722047e+00,
         3.58251869e+00, 4.02781691e+00, 4.47311512e+00, 4.91841334e+00,
         5.36371156e+00, 5.80900977e+00, 6.25430799e+00, 6.69960621e+00,
         7.14490442e+00, 7.59020264e+00, 8.03550086e+00, 8.48079907e+00,
         8.92609729e+00, 9.37139551e+00, 9.81669372e+00, 1.02619919e+01,
         1.07072902e+01, 1.11525884e+01, 1.15978866e+01, 1.20431848e+01,
         1.24884830e+01, 1.29337812e+01, 1.33790795e+01, 1.38243777e+01,
         1.42696759e+01, 1.47149741e+01, 1.51602723e+01, 1.56055705e+

In [8]:
[plt.hist(ang, alpha = 0.5, bins = 50) for ang in dist2d_mean]

[(array([577., 879., 503., 308., 190., 196., 182., 151., 149.,  94.,  75.,
         111., 125., 149., 107.,  77.,  78.,  93.,  32.,  30.,  24.,  25.,
          30.,  29.,  35.,  36.,  43.,  51.,  21.,  20.,  20.,  18.,  16.,
          15.,  12.,  15.,  20.,  20.,  17.,  19.,  20.,  16.,  13.,  13.,
          14.,  17.,  18.,  21.,  45.,  31.]),
  array([8.92525159e-03, 4.89866749e-01, 9.70808246e-01, 1.45174974e+00,
         1.93269124e+00, 2.41363274e+00, 2.89457423e+00, 3.37551573e+00,
         3.85645723e+00, 4.33739873e+00, 4.81834022e+00, 5.29928172e+00,
         5.78022322e+00, 6.26116471e+00, 6.74210621e+00, 7.22304771e+00,
         7.70398921e+00, 8.18493070e+00, 8.66587220e+00, 9.14681370e+00,
         9.62775519e+00, 1.01086967e+01, 1.05896382e+01, 1.10705797e+01,
         1.15515212e+01, 1.20324627e+01, 1.25134042e+01, 1.29943457e+01,
         1.34752872e+01, 1.39562287e+01, 1.44371702e+01, 1.49181117e+01,
         1.53990532e+01, 1.58799947e+01, 1.63609362e+01, 1.68418777e+

In [415]:

def intersect_all_cams(frames,cam,intersected,tol = 1):
    for cam in range(4):
        intersected = Utils.intersection_per_cam(frames, cam, intersected, tol=tol) 
    return intersected

frames_list=[]
mov = 36
for frame_num in range(1031,1032,3):
    interest_points_path = f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/evaluation/points/mov{mov}'

    image_path =  f'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov{mov}_2023_08_09_60ms/'
    frame = FlyOutput(image_path,frame_num,input_dir,output_angles_weights,frame0,iteration,file_name,frames_dict = frames)
    frame.intersect_projections
    interest_points = frame.load_parimiter(interest_points_path)
    frame.add_interest_points_and_triangulate(interest_points)
    frame.define_wings_interest_points()
    frame.calc_wing_le_te(num_of_bins = 20,perc_wing_for_le = 1,wing_length_snip = 0.15)
    frame.fit_interest_and_gs()
    frame.calculate_dist_interest_gs()
    frames_list.append(frame)

prev_length = 0 
frames_interest_points = {}
for key in interest_points.keys():
    frames_interest_points[key] = list(range(prev_length,prev_length + len(interest_points[key]))) 
    prev_length += len(interest_points[key])






IndexError: list index out of range

In [200]:
frame.dist_3d_interest_to_gs

array([0.00100013, 0.00099524, 0.00099041, ..., 0.00088319, 0.00088112,
       0.00087907])

In [235]:
hist_points_dist_3d_interest_to_gs = np.hstack([frame.dist_3d_interest_to_gs for frame in frames_list ])
hist_points_dist_2d_interest_to_gs = np.hstack([frame.dist_2d_interest_to_gs for frame in frames_list ])
hist_points_dist_3d_gs_to_interest = np.hstack([frame.dist_3d_gs_to_interest for frame in frames_list ])
hist_points_dist_2d_gs_to_interest = np.hstack([frame.dist_2d_gs_to_interest for frame in frames_list ])

plt.figure()
plt.title(f' gs from interest - 3D distance mean {np.mean(hist_points_dist_3d_gs_to_interest.flatten()*1000):.2f} std{np.std(hist_points_dist_3d_gs_to_interest.flatten()*1000):.2f}')
plt.hist(1000*hist_points_dist_3d_gs_to_interest.flatten(),bins = 50)
plt.xlabel(f'3D distance [mm]')


plt.figure()
plt.title(f'interest from gs - 3D distance mean {np.mean(hist_points_dist_3d_interest_to_gs.flatten()*1000):.2f} std{np.std(hist_points_dist_3d_interest_to_gs.flatten()*1000):.2f}')
plt.hist(1000*hist_points_dist_3d_interest_to_gs.flatten(),bins = 50)
plt.xlabel(f'3D distance  [mm]')


plt.figure()
plt.title(f' gs from interest - reprojection error mean {np.mean(hist_points_dist_2d_interest_to_gs.flatten()):.2f} std{np.std(hist_points_dist_2d_interest_to_gs.flatten()):.2f}')
plt.hist(hist_points_dist_2d_interest_to_gs.flatten(),bins = 50)
plt.xlabel(f'Reprojection error [pixel]')


plt.figure()
plt.title(f'interest from gs - reprojection error mean {np.mean(hist_points_dist_2d_gs_to_interest.flatten()):.2f} std{np.std(hist_points_dist_2d_gs_to_interest.flatten()):.2f}')
plt.hist(hist_points_dist_2d_gs_to_interest.flatten(),bins = 50)
plt.xlabel(f'Reprojection error [pixel]')

Text(0.5, 0, 'Reprojection error [pixel]')

In [13]:
import plotly.graph_objects as go
import numpy as np


color_list = ['lime','crimson','magenta','magenta','dodgerblue','blue','blue','black','orange']
name_list = ['body','right wing','right wing le','right wing te','left wing','left wing le','left wing te','Ground truth','gaussian points']
size_list = [2,2,4,4,2,4,4,5,5]
framestart = 1620
frame_end = 1621
frames = range(framestart, frame_end)
output_path = f'{path_output}/{model_name}/animated_plot.html'

xyz_all_frames = np.vstack([frame.xyz_rotated for frame in frames_list])

# === HELPERS ===

def create_scatter3d(xyz, color,name,size = 2):
    """Create a single 3D scatter trace for a specific part."""
    return go.Scatter3d(
        x=xyz[:, 0],
        y=xyz[:, 1],
        z=xyz[:, 2],
        mode="markers",
        name = name,
        marker=dict(size=size, opacity=1, color=color, colorscale='gray'),
    )


def get_global_bounds(xyz_list):
    """Compute global min and max coordinates over all frames for consistent axis scaling."""
    return np.min(xyz_list, axis=0), np.max(xyz_list, axis=0)


def create_frame(parts_list, color_list,size_list, frame_name,name_list):
    """Create one animation frame with all parts for a given timestep."""
    data = [
        create_scatter3d(part, color,name,size)
        for part, color,size,name in zip(parts_list, color_list, size_list,name_list)
    ]
    return go.Frame(data=data, name=frame_name)


def create_play_pause_buttons():
    """Return Play/Pause button definitions for animation."""
    return [
        {
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}],
                    "label": "Play",
                    "method": "animate",
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                    "label": "Pause",
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 87},
            "showactive": False,
            "type": "buttons",
            "x": 0.1,
            "xanchor": "right",
            "y": 0,
            "yanchor": "top",
        }
    ]

def create_slider(frame_nums):
    """Create a slider using actual frame numbers."""
    return [
        {
            "active": 0,
            "steps": [
                {
                    "args": [[str(i)], {"frame": {"duration": 100, "redraw": True}, "mode": "immediate"}],
                    "label": str(frame_num),
                    "method": "animate",
                }
                for i, frame_num in enumerate(frame_nums)
            ],
        }
    ]


# === MAIN FUNCTION ===

def create_3d_animation(frames_list, color_list,xyz_all_frames,size_list,name_list):
    """Build and show the 3D animation."""
    frame_nums = [f.frame_num for f in frames_list[framestart - frame0:frame_end - frame0]]
    min_xyz, max_xyz = get_global_bounds(xyz_all_frames)

    # Initial frame data
    intial_parts = [frames_list[0].body,frames_list[0].right_wing,frames_list[0].right_wing_le,frames_list[0].right_wing_te,frames_list[0].left_wing,frames_list[0].left_wing_le,frames_list[0].left_wing_te, frames_list[0].rotated_points_3d]
    initial_data = [
        create_scatter3d(part, color,name,size)
        for  part,color,size,name in zip(intial_parts, color_list,size_list,name_list)
    ]
    bounding_box_trace = go.Scatter3d(
    x=[min_xyz[0], max_xyz[0]],
    y=[min_xyz[1], max_xyz[1]],
    z=[min_xyz[2], max_xyz[2]],
    mode='markers',
    marker=dict(size=0.1, color='rgba(0,0,0,0)'),
    showlegend=False
    )
    initial_data.append(bounding_box_trace)
    # Create frames for animation
    frames_data = [
        create_frame([xyz_frame.body,xyz_frame.right_wing,xyz_frame.right_wing_le,xyz_frame.right_wing_te,xyz_frame.left_wing,xyz_frame.left_wing_le,xyz_frame.left_wing_te, xyz_frame.rotated_points_3d], color_list,size_list, str(i),name_list)
        for i, xyz_frame in enumerate(frames_list[framestart - frame0:frame_end - frame0])
    ]

    # Build full figure
    fig = go.Figure(
        data=initial_data,
        layout=go.Layout(
            scene=dict(
                xaxis_title="X",
                yaxis_title="Y",
                zaxis_title="Z",
            ),
            updatemenus=create_play_pause_buttons(),
            sliders=create_slider(frame_nums),
        ),
        frames=frames_data,
    )

    fig.show()
    fig.write_html(output_path)
    print(f"Saved animation to: {output_path}")


create_3d_animation([all_frames[0]], color_list,xyz_all_frames,size_list,name_list)




    # point_3d_per_frame.append(np.vstack(points_3d))
    # gaussians_interest_points.append(gaussian_points)

Saved animation to: D:/Documents/gaussian_model_output//fly_angle_sweeps_roll_yaw_nom_all_frames_1000/animated_plot.html


In [527]:
    import itertools
    yaw_grid = np.hstack(np.arange(0,360,30))
    roll_grid = np.hstack(np.arange(-30,30,10))
    psi_grid = np.hstack((np.arange(-160,0,30)))
    phi_grid = np.hstack((np.arange(-90,90,30)))

    roll_yaw = list(itertools.product(yaw_grid,roll_grid))
    psi_phi = list(itertools.product(psi_grid,phi_grid))
    # roll_yaw = roll_grid
    # pitch_grid = np.hstack((0.0,np.arange(-20,0,5),np.arange(5,20,5)))
    roll_yaw = roll_yaw + psi_phi

In [530]:
108-36

72

In [54]:
frame = 370
color = frames_list[frame - frame0].color
idx_part = frames_list[frame - frame0].idx_parts


grayscale = (color[:,0] - color[:,0].min()) / (color[:,0].max() - color[:,0].min())

opacity = frames_list[frame - frame0].opacity * grayscale
# grayscale = grayscale[grayscale <1]

frame = frames_list[frame-frame0]
fig = go.Figure()
Plotters.scatter3d(fig,frame.body,'green',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing,opacity[idx_part[1]],2,'right wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing,opacity[idx_part[2]],2,'left wing',show_colorbar = False)
Plotters.scatter3d(fig,frame.rotated_points_3d[7:8,:],'blue',5,'interest',show_colorbar = False)
# Plotters.scatter3d(fig,frames_list[frame-frame0].gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.right_wing_te,'cyan',4,'te',show_colorbar = False)

Plotters.scatter3d(fig,frame.left_wing_le,'magenta',4,'le',show_colorbar = False)
Plotters.scatter3d(fig,frame.left_wing_te,'cyan',4,'te',show_colorbar = False)


t = np.linspace(-0.001, 0.0015, 100)  # Small range since your data seems very small-scale

r_line_points = frame.right_wing_origin + t[:, np.newaxis] * frame.right_wing_span
l_line_points = frame.left_wing_origin + t[:, np.newaxis] * frame.left_wing_span
idx_closest = np.unique([np.argsort(frame.dist_points(point,frame.body))[0:10] for point in r_line_points])
# idx_origin = np.argmin(np.dot(frame.body[idx_closest,:],frame.right_wing_direction))
idx_origin = np.argmax([np.min(np.dot(frame.body[idx_closest,:],r_line_points.T)) for idx_closest in idx_closest])
idx_origin


Plotters.scatter3d(fig,r_line_points,'orange',3,'wing',show_colorbar = False)
Plotters.scatter3d(fig,l_line_points,'orange',3,'wing',show_colorbar = False)
# Plotters.scatter3d(fig,np.atleast_2d(frame.body[idx_closest[idx_origin],:]),'black',10,'wing',show_colorbar = False)

# Plotters.scatter3d(fig,frame.gaussian_closest_to_interest,'orange',5,'interest gauss',show_colorbar = False)
# Plotters.scatter3d(fig,frame.body_interest_gaussian,'orange',5,'interest gauss',show_colorbar = False)
# dot_on_le = np.dot(-frame.right_wing_direction,frame.right_wing_le.T)
# length = (np.max(dot_on_le) - np.min(dot_on_le))
# lt20p = frame.right_wing_le[(dot_on_le  - np.min(dot_on_le))> length*0.1,:]
# Plotters.scatter3d(fig,frame.rotated_points_3d[8:16],'orange',3,'interest gauss',show_colorbar = False)
# Plotters.scatter3d(fig, np.vstack((np.mean(frame.bottom,axis = 0) - frame.xbody*2/1000,np.mean(frame.body,axis = 0) + frame.xbody*2/1000)),'black',5,'x',mode = 'markers+lines') 
# Plotters.scatter3d(fig, lt20p,'magenta',5,'x') 
# Plotters.scatter3d(fig,bod_ax_top,'black',3,'body',show_colorbar = False)

# Plotters.scatter3d(fig, np.atleast_2d(frame.interest_on_xbody),'black',10,'inter_on_body',mode = 'markers+lines') 
# Plotters.scatter3d(fig, frame.rotated_points_3d[16:,:],'black',10,'inter',mode = 'markers+lines') 


# fig.show()

# ax = None
# ax = Plotters.plot_projections(frame.interest_points_3d[:,:],frame.frames,color = 'magenta',ax = ax, size = 5)
# ax = Plotters.plot_projections(frame.gaussian_closest_to_interest_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)
# # ax = Plotters.plot_projections(frame.body_interest_gaussian_ew[:,:],frame.frames,color = 'orange',ax = ax, size = 5)

# ax = None
# ax = Plotters.plot_projections(frame.right_wing_ew[:,:],frame.frames,color = 'red',ax = ax, size = 5)
# ax = Plotters.plot_projections(frame.left_wing_ew[:,:],frame.frames,color = 'blue',ax = ax, size = 5)


AttributeError: 'FlyOutput' object has no attribute 'dist_points'

In [ ]:
dot_on_le = np.dot(frame.right_wing_direction,frame.right_wing_le.T)
length = (np.max(dot_on_le) - np.min(dot_on_le))
lt20p = dot_on_le[dot_on_le > length*0.2,:]

0.002344680935456922

In [39]:
idx_origin = np.argmin([np.min(np.dot(frame.body[idx_closest,:],r_line_points.T)) for idx_closest in idx_closest])
idx_origin

89

In [ ]:
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)

plt.figure(),plt.plot(pts_on_nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
plt.figure()
plt.plot((pts_on_nrml - mean)/std)


In [11]:
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)
plt.figure()
plt.plot(points_2d[:,0],points_2d[:,1])

plt.figure()
plt.plot(points_2d)

In [ ]:

wing_gs,interest_rw,interest_lw = frame.zsocre_ol_calc_indices()


In [21]:
from scipy.signal import savgol_filter


frame = frames_list[-1]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)


x = savgol_filter(points_2d[:,0], 15, 2)
y = savgol_filter(points_2d[:,1], 15, 2)


plt.scatter(points_2d[:,0],points_2d[:,1])

plt.scatter(x,y)

In [ ]:
frame = frames_list[-1]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
nrml = np.cross(frame.right_wing_span,frame.right_wing_chord)
pts_on_nrml = np.dot(wing_bound,nrml)
std = np.std(pts_on_nrml)
mean = np.mean(pts_on_nrml)
wing_bound = wing_bound[((pts_on_nrml - mean)/std) < 1.5]

indices_wing_bound = frame.cyclic_sort(wing_bound,frame.right_wing_span,frame.right_wing_chord)
points_2d = Utils.project_to_plane(wing_bound[indices_wing_bound], np.mean(wing_bound[indices_wing_bound],axis = 0), frame.right_wing_span,frame.right_wing_chord)



wing_bound = wing_bound[indices_wing_bound]
pts_to_fit = [wing_bound[k:k+3] for k in range(0,wing_bound.shape[0],3)]

fit = []
for pts in pts_to_fit[:-1]: 

    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning
    p = np.polyfit(t, pts, 2)
    t_fit = np.linspace(t[0], t[-1], 1000)
    fit_xyz = np.vstack([np.polyval(p, t_fit) for p in p.T]).T
    fit.append(fit_xyz)
fit = np.vstack(fit)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot(wing_bound[:,0], wing_bound[:,1], wing_bound[:,2], 'ro', label='Original points')
ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
ax.legend()
plt.show()

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned



In [26]:
fit.shape

(21000, 3)

In [ ]:
from math import atan2
normal_to_wing = np.cross(frame.right_wing_span,frame.right_wing_chord)

wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))



frameidx = 4
frame = frames_list[frameidx]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
interest = frames_list[frameidx].rotated_points_3d[frame.interest_right_wing_boundry,:]



points_2d = Utils(wing_bound, np.mean(wing_bound,axis = 0), frame.right_wing_span, frame.right_wing_chord)
indices = Utils.rotational_sort(points_2d, np.mean(points_2d,axis = 0), clockwise=True)
# indices2 = rotational_sort(points_2d, np.mean(points_2d,axis = 0), clockwise=True)


wing_bound
fig = go.Figure()
# Plotters.scatter3d(fig,frame.right_wing,'red',3,'body',show_colorbar = False)
# Plotters.scatter3d(fig,interest[indices,:],'black',3,'interest',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,wing_bound[indices2,:],'blue',3,'gs',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,np.atleast_2d(frame.right_wing_origin),'blue',10,'gs',show_colorbar = False,mode='markers+lines')

fig.show()


In [30]:
from math import atan2

def argsort(seq):
    #http://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python/3382369#3382369
    #by unutbu
    #https://stackoverflow.com/questions/3382352/equivalent-of-numpy-argsort-in-basic-python 
    # from Boris Gorelik
    return sorted(range(len(seq)), key=seq.__getitem__)

def rotational_sort(list_of_xy_coords, centre_of_rotation_xy_coord, clockwise=True):
    cx,cy=centre_of_rotation_xy_coord
    angles = [atan2(x-cx, y-cy) for x,y in list_of_xy_coords]
    indices = argsort(angles)
    # if clockwise:
    #     return [list_of_xy_coords[i] for i in indices]
    # else:
    #     return [list_of_xy_coords[i] for i in indices[::-1]]
    return indices

frameidx = 4
frame = frames_list[frameidx]
wing_bound = np.vstack((frame.right_wing_le,frame.right_wing_te))
interest = frames_list[frameidx].rotated_points_3d[frame.interest_right_wing_boundry,:]
indices = rotational_sort(interest[:,[1,2]], np.mean(interest[:,[1,2]],axis = 0), clockwise=True)
indices2 = rotational_sort(wing_bound[:,[1,2]], np.mean(wing_bound[:,[1,2]],axis = 0), clockwise=True)


wing_bound
fig = go.Figure()
# Plotters.scatter3d(fig,frame.right_wing,'red',3,'body',show_colorbar = False)
Plotters.scatter3d(fig,interest[indices,:],'black',3,'interest',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,wing_bound[indices2,:],'blue',3,'gs',show_colorbar = False,mode='markers+lines')
Plotters.scatter3d(fig,np.atleast_2d(frame.right_wing_origin),'blue',10,'gs',show_colorbar = False,mode='markers+lines')

fig.show()


In [59]:
import numpy as np
import matplotlib.pyplot as plt

# Your 3D points (N x 3)
points = interest[indices,:]
# points = wing_bound[indices2,:]

pts_to_fit = [points[k:k+3] for k in range(0,points.shape[0],3)]
# pts_to_fit = [points[k:k+5] for k in range(0,points.shape[0],5)]


fit = []
for pts in pts_to_fit[:-1]: 
    # Step 1: Generate parameter t (cumulative distance)
    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning

    # Step 2: Fit a 2nd degree polynomial separately for x(t), y(t), z(t)
    degree = 2
    px = np.polyfit(t, pts[:,0], degree)
    py = np.polyfit(t, pts[:,1], degree)
    pz = np.polyfit(t, pts[:,2], degree)

    # To evaluate the fitted curve:
    t_fit = np.linspace(t[0], t[-1], 1000)
    x_fit = np.polyval(px, t_fit)
    y_fit = np.polyval(py, t_fit)
    z_fit = np.polyval(pz, t_fit)
    fit.append(np.vstack((x_fit, y_fit, z_fit)))
fit = np.hstack(fit).T
# Plotting
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.plot(points[:,0], points[:,1], points[:,2], 'ro', label='Original points')
# ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
# ax.legend()
# plt.show()


In [60]:
import numpy as np
import matplotlib.pyplot as plt

# Your 3D points (N x 3)
points = interest[indices,:]
points = wing_bound[indices2,:]

pts_to_fit = [points[k:k+3] for k in range(0,points.shape[0],3)]
# pts_to_fit = [points[k:k+5] for k in range(0,points.shape[0],5)]


fit2 = []
for pts in pts_to_fit[:-1]: 
    # Step 1: Generate parameter t (cumulative distance)
    dists = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    t = np.insert(np.cumsum(dists), 0, 0)  # insert 0 at the beginning

    # Step 2: Fit a 2nd degree polynomial separately for x(t), y(t), z(t)
    degree = 2
    px = np.polyfit(t, pts[:,0], degree)
    py = np.polyfit(t, pts[:,1], degree)
    pz = np.polyfit(t, pts[:,2], degree)

    # To evaluate the fitted curve:
    t_fit = np.linspace(t[0], t[-1], 1000)
    x_fit = np.polyval(px, t_fit)
    y_fit = np.polyval(py, t_fit)
    z_fit = np.polyval(pz, t_fit)
    fit2.append(np.vstack((x_fit, y_fit, z_fit)))
fit2 = np.hstack(fit2).T
# Plotting
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.plot(points[:,0], points[:,1], points[:,2], 'ro', label='Original points')
# ax.plot(fit[:,0], fit[:,1], fit[:,2], 'm-', label='Fitted 2nd order curve')
# ax.legend()
# plt.show()


c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarning:

Polyfit may be poorly conditioned

c:\Users\Roni\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3460: RankWarni

In [61]:

def dist_points(x1,x2):
    return np.sqrt(np.sum((x1 - x2)**2, axis = 1))

gaussian_closest_to_interest = np.vstack((fit2[np.argmin(dist_points(fit2,point)),:] for point in fit))
fitted_closest_to_gauss = np.vstack((fit[np.argmin(dist_points(fit,point)),:] for point in fit2))


plt.figure()
plt.hist(1000*dist_points(gaussian_closest_to_interest,fit))

plt.figure()
plt.hist(1000*dist_points(fitted_closest_to_gauss,fit2))

C:\Users\Roni\AppData\Local\Temp\ipykernel_14272\387352763.py:4: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.

C:\Users\Roni\AppData\Local\Temp\ipykernel_14272\387352763.py:5: FutureWarning:

arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.



(array([3399., 5420., 5358., 4233., 2123.,  666.,  294.,  169.,  161.,
         177.]),
 array([0.00934659, 0.03537276, 0.06139894, 0.08742511, 0.11345128,
        0.13947745, 0.16550363, 0.1915298 , 0.21755597, 0.24358214,
        0.26960831]),
 <BarContainer object of 10 artists>)

In [ ]:


def dist_points(x1,x2):
    return np.sqrt(np.sum((x1 - x2)**2, axis = 1))

gaussian_closest_to_interest = np.vstack((fit[np.argmin(dist_points(fit,point)),:] for point in wing_bound))


    def closest_point_to_interest_boundary(self,wing_boundary,points):   

        gaussian_closest_to_interest = np.vstack((wing_boundary[np.argmin(self.dist_points(wing_boundary,point)),:] for point in points))
        gaussian_closest_to_interest_ew = (self.ew_to_lab.T @ np.vstack(gaussian_closest_to_interest).T).T
        dist_gaus_interest = self.dist_points(gaussian_closest_to_interest[1:,:],gaussian_closest_to_interest[0:-1,:])
        dist_interest = self.dist_points(points[1:,:],points[0:-1,:])   
        return  gaussian_closest_to_interest,gaussian_closest_to_interest_ew,dist_gaus_interest,dist_interest
